In [ ]:
import pylhe
import math
import sys
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

import hist
from hist import Hist

for fileVal in [2,3,4,5,6,7,8,9]:
    file_name = f'/home/joseph/MG_tutorial/MG5_aMC_v3_6_7/ufo1/Events/run_0{fileVal}_decayed_1/unweighted_events.lhe'

    eventCount = 1
    bothCount04 = 0
    anyCount04 = 0
    bothCount1 = 0
    anyCount1=0
    threshold04 = 0.4
    threshold1 = 1
    successCount = 0
    chosenCorrectlyCount = 0
    sameMomCount = 0
    extraUCount = 0
    bsMet = 0
    bothMet = 0


    trueL = []

    for event in pylhe.read_lhe(file_name):

        particle_list = event.particles
        event_info = event.eventinfo

        pTMax = 0
        indpTMax = 0
        for i in range(len(particle_list)):
            pp = particle_list[i]
            if abs(pp.id) == 1 or abs(pp.id) == 2 or abs(pp.id) == 3:
                pTcheck = np.sqrt(pp.px**2+pp.py**2)
                if pTcheck>pTMax:
                    pTMax = pTcheck
                    indpTMax = i

        maxPart = particle_list[indpTMax]

        thetaMP = np.arctan(maxPart.pz/np.sqrt(maxPart.px**2+maxPart.py**2))
        phiMP = np.arctan(maxPart.py/maxPart.px)
        if(np.tan(thetaMP/2) == 0):
            etaMP = 999999 # Effectively infinity, to avoid python evaluating log(0)
        else:
            etaMP = -1*np.log(np.tan(thetaMP/2))

        delRHigh = 10000000
        indHigh = 0
        delRHigh2 = 10000000
        indHigh2 = 0


        for i in range(len(particle_list)):
            if i != indpTMax:
                pp = particle_list[i]
                if (abs(pp.id) == 1 or abs(pp.id) == 2 or abs(pp.id) == 3) and pp.px != 0 and pp.py != 0:
                    theta = np.arctan(pp.pz/np.sqrt(pp.px**2+pp.py**2))
                    phi = np.arctan(pp.py/pp.px) if pp.px != 0 else math.pi/2*abs(pp.py)/pp.py
                    if(np.tan(theta/2) == 0):
                        eta = 999999 # Effectively infinity, to avoid python evaluating log(0)
                    else:
                        eta = -1*np.log(np.tan(theta/2))
                    delR = np.sqrt((phiMP-phi)**2+(etaMP-eta)**2)

                    if delR < delRHigh:
                        delRHigh2 = delRHigh
                        indHigh2 = indHigh
                        delRHigh = delR
                        indHigh = i
                    elif delR<delRHigh2:
                        delRHigh2 = delR
                        indHigh2 = i

        dSet1 = [maxPart, particle_list[indHigh], particle_list[indHigh2]]

        pTMax = 0
        indpTMaxSecond = 0
        for i in range(len(particle_list)):
            pp = particle_list[i]
            if (abs(pp.id) == 1 or abs(pp.id) == 2 or abs(pp.id) == 3) and i != indpTMax and i!= indHigh and i!=indHigh2:
                pTcheck = np.sqrt(pp.px**2+pp.py**2)
                if pTcheck>pTMax:
                    pTMax = pTcheck
                    indpTMaxSecond = i

        maxPartSecond = particle_list[indpTMaxSecond]

        thetaMP2 = np.arctan(maxPartSecond.pz/np.sqrt(maxPartSecond.px**2+maxPartSecond.py**2))
        phiMP2 = np.arctan(maxPartSecond.py/maxPartSecond.px)
        if(np.tan(thetaMP/2) == 0):
            etaMP2 = 999999 # Effectively infinity, to avoid python evaluating log(0)
        else:
            etaMP2 = -1*np.log(np.tan(thetaMP/2))

        delRHigh = 10000000
        indHighSecond = 0
        delRHigh2 = 10000000
        indHigh2Second = 0


        for i in range(len(particle_list)):
            if i != indpTMaxSecond and i!= indHigh and i!=indHigh2 and i != indpTMax:
                pp = particle_list[i]
                if (abs(pp.id) == 1 or abs(pp.id) == 2 or abs(pp.id) == 3) and pp.px != 0 and pp.py != 0:
                    theta = np.arctan(pp.pz/np.sqrt(pp.px**2+pp.py**2))
                    phi = np.arctan(pp.py/pp.px) if pp.px != 0 else math.pi/2*abs(pp.py)/pp.py
                    if(np.tan(theta/2) == 0):
                        eta = 999999 # Effectively infinity, to avoid python evaluating log(0)
                    else:
                        eta = -1*np.log(np.tan(theta/2))
                    delR = np.sqrt((phiMP2-phi)**2+(etaMP2-eta)**2)

                    if delR < delRHigh:
                        delRHigh2 = delRHigh
                        indHigh2Second = indHighSecond
                        delRHigh = delR
                        indHighSecond = i
                    elif delR<delRHigh2:
                        delRHigh2 = delR
                        indHigh2Second = i

        dSet2 = [maxPartSecond, particle_list[indHighSecond], particle_list[indHigh2Second]]

        org = bsMet

        motherOne = dSet1[0].mother1
        if all(pp.mother1 == motherOne for pp in dSet1):
            bsMet += 1
        motherTwo = dSet2[0].mother1
        if all(pp.mother1 == motherTwo for pp in dSet2):
            bsMet += 1

        if bsMet == org+2:
            bothMet +=1 

        #print("This is event", eventCount)
        #print(len(particle_list), "particles were made in this event")

        otherUp1 = particle_list[4]
        otherUp2 = particle_list[6]

        thetaO1 = np.arctan(otherUp1.pz/np.sqrt(otherUp1.px**2+otherUp1.py**2))
        phiO1 = np.arctan(otherUp1.py/otherUp1.px)
        if(np.tan(thetaO1/2) == 0):
            etaO1 = 999999 # Effectively infinity, to avoid python evaluating log(0)
        else:
            etaO1 = -1*np.log(np.tan(thetaO1/2))

        thetaO2 = np.arctan(otherUp2.pz/np.sqrt(otherUp2.px**2+otherUp2.py**2))
        phiO2 = np.arctan(otherUp2.py/otherUp2.px)
        if(np.tan(thetaO2/2) == 0):
            etaO2 = 999999 # Effectively infinity, to avoid python evaluating log(0)
        else:
            etaO2 = -1*np.log(np.tan(thetaO2/2))


        n1One = particle_list[8:11]
        maxIndOne = 0
        delROne = []
        delROneElse = []
        n1Two = particle_list[11:]
        maxIndTwo = 0
        delRTwo = []
        delRTwoElse = []

        pTMax = 0

        for i in range(len(n1One)):
            pT = (np.sqrt(n1One[i].px**2+n1One[i].py**2))
            if pT > pTMax:
                pTMax = pT
                maxIndOne = i

        ppM1 = n1One[maxIndOne]

        pTMax = 0

        for j in range(len(n1Two)):
            pT = (np.sqrt(n1Two[j].px**2+n1Two[j].py**2))
            if pT > pTMax:
                pTMax = pT
                maxIndTwo = j

        ppM2 = n1Two[maxIndTwo]

        ppTMax1 = 0
        ppTMax2 = 0
        pppInd1 = 0
        pppInd2 = 0
        for j in range(len(particle_list)):
            ppp = particle_list[j]
            if (abs(ppp.id) == 2 or abs(ppp.id) == 1 or abs(ppp.id) == 3):
                ppT = (np.sqrt(ppp.px**2+ppp.py**2))
                if ppT > ppTMax1:
                    ppTMax2 = ppTMax1
                    ppTMax1 = ppT
                    pppInd2 = pppInd1
                    pppInd1 = j
                elif ppT > ppTMax2:
                    ppTMax2 = ppT
                    pppInd2 = j

        if (pppInd1 == maxIndOne+8 and pppInd2 == maxIndTwo+11) or (pppInd1 == maxIndTwo+11 and pppInd2 == maxIndOne+8):
            chosenCorrectlyCount +=1
        elif (particle_list[pppInd1].mother1 == particle_list[pppInd2].mother1):
            sameMomCount +=1
        elif particle_list[pppInd1].mother1<6 or particle_list[pppInd2].mother1<6:
            if particle_list[pppInd1].mother1<6:
                extraUCount += 1
            if particle_list[pppInd2].mother1<6:
                extraUCount += 1
            


        
        #theta = arctan(pz/pt)
        #phi = arctan(py/px)
        #eta = - ln tan (theta/2)
        #delR = sqrt(delPhi**2+delEta**2)

        thetaM2 = np.arctan(ppM2.pz/np.sqrt(ppM2.px**2+ppM2.py**2))
        phiM2 = np.arctan(ppM2.py/ppM2.px)
        if(np.tan(thetaM2/2) == 0):
            etaM2 = 999999 # Effectively infinity, to avoid python evaluating log(0)
        else:
            etaM2 = -1*np.log(np.tan(thetaM2/2))    

        thetaM1 = np.arctan(ppM1.pz/np.sqrt(ppM1.px**2+ppM1.py**2))
        phiM1 = np.arctan(ppM1.py/ppM1.px)
        if(np.tan(thetaM1/2) == 0):
            etaM1 = 999999 # Effectively infinity, to avoid python evaluating log(0)
        else:
            etaM1 = -1*np.log(np.tan(thetaM1/2))


        ind = maxIndOne + 8
        dtrs = [particle_list[ind]]
        while len(dtrs) < 3:
            indAdd = 0
            delRmin = 100000000
            for i in range(len(particle_list)):
                if i != ind:    
                    pp = particle_list[i]
                    if (abs(pp.id) == 2 or abs(pp.id) == 1 or abs(pp.id) == 3) and pp.px != 0 and pp.py != 0:
                        theta = np.arctan(pp.pz/np.sqrt(pp.px**2+pp.py**2))
                        phi = np.arctan(pp.py/pp.px) if pp.px != 0 else math.pi/2*abs(pp.py)/pp.py
                        if(np.tan(theta/2) == 0):
                            eta = 999999 # Effectively infinity, to avoid python evaluating log(0)
                        else:
                            eta = -1*np.log(np.tan(theta/2))
                        delR = np.sqrt((phiM1-phi)**2+(etaM1-eta)**2)
                        if delR < delRmin:
                            delRmin = delR
                            indAdd = i

            dtrs += [particle_list[indAdd]]

        first_mother = dtrs[0].mother1
        if all(pp.mother1==first_mother for pp in dtrs):
            #print("WORKED")
            successCount +=1
            #print(sucessCount)

        ind2 = maxIndTwo + 11
        dtrs = []
        dtrs = [particle_list[ind2]]
        while len(dtrs) < 3:
            indAdd = 0
            delRmin = 100000000
            for i in range(len(particle_list)):
                if i != ind2:    
                    pp = particle_list[i]
                    if (abs(pp.id) == 2 or abs(pp.id) == 1 or abs(pp.id) == 3) and pp.px != 0 and pp.py != 0:
                        theta = np.arctan(pp.pz/np.sqrt(pp.px**2+pp.py**2))
                        phi = np.arctan(pp.py/pp.px) if pp.px != 0 else math.pi/2*abs(pp.py)/pp.py
                        if(np.tan(theta/2) == 0):
                            eta = 999999 # Effectively infinity, to avoid python evaluating log(0)
                        else:
                            eta = -1*np.log(np.tan(theta/2))
                        delR = np.sqrt((phiM2-phi)**2+(etaM2-eta)**2)
                        if delR < delRmin:
                            delRmin = delR
                            indAdd = i

            dtrs += [particle_list[indAdd]]

        first_mother = dtrs[0].mother1
        if all(pp.mother1==first_mother for pp in dtrs):
            #print("WORKED")
            successCount +=1
            #print(sucessCount)


        for i in range(len(n1One)):
            if i != maxIndOne:
                other1 = n1One[i]
                thetaO = np.arctan(other1.pz/np.sqrt(other1.px**2+other1.py**2))
                phiO = np.arctan(other1.py/other1.px)
                if(np.tan(thetaO/2) == 0):
                    etaO = 999999 # Effectively infinity, to avoid python evaluating log(0)
                else:
                    etaO = -1*np.log(np.tan(thetaO/2))

                delROne+=[np.sqrt((phiM1-phiO)**2+(etaM1-etaO)**2)]
                delRTwoElse += [np.sqrt((phiM2-phiO)**2+(etaM2-etaO)**2)]

        for i in range(len(n1Two)):
            if i != maxIndTwo:
                other = n1Two[i]
                thetaO = np.arctan(other.pz/np.sqrt(other.px**2+other.py**2))
                phiO = np.arctan(other.py/other.px)
                if(np.tan(thetaO/2) == 0):
                    etaO = 999999 # Effectively infinity, to avoid python evaluating log(0)
                else:
                    etaO = -1*np.log(np.tan(thetaO/2))

                delRTwo+=[np.sqrt((phiM2-phiO)**2+(etaM2-etaO)**2)]
                delROneElse += [np.sqrt((phiM1-phiO)**2+(etaM1-etaO)**2)]

        delRTwoElse += [np.sqrt((phiM2-phiO1)**2+(etaM2-etaO1)**2)]
        delRTwoElse += [np.sqrt((phiM2-phiO2)**2+(etaM2-etaO2)**2)]
        delROneElse += [np.sqrt((phiM2-phiO1)**2+(etaM2-etaO1)**2)]
        delROneElse += [np.sqrt((phiM2-phiO2)**2+(etaM2-etaO2)**2)]

        One04 = False
        Two04 = False
        One1 = False
        Two1 = False

        if all(val<threshold04 for val in delROne) and not any(val<threshold04 for val in delROneElse):
            One04 = True
        
        if all(val<threshold04 for val in delRTwo) and not any(val<threshold04 for val in delRTwoElse):
            Two04 = True

        if One04 or Two04:
            anyCount04 +=1
        if One04 and Two04:
            bothCount04 +=1

        if all(val<threshold1 for val in delROne) and not any(val<threshold1 for val in delROneElse):
            One1 = True
        
        if all(val<threshold1 for val in delRTwo) and not any(val<threshold1 for val in delRTwoElse):
            Two1 = True

        if One1 or Two1:
            anyCount1 +=1
        if One1 and Two1:
            bothCount1 +=1

        

        

        eventCount+=1

        """
        for pp in particle_list:
                
                #indexes 8,9,10 and 11,12,13 come from the n1s from ur and ur~
                if pp.mother1 == 6:
                    


                
                if (pp.id == 2 or pp.id==-2):
                    if pp.mother1 == 3:
                        uE = pp.e
                        upx = pp.px
                        upy = pp.py
                        upz = pp.pz
                        u4 += [(uE, upx, upy, upz,)]
                    print("Up")

                if (pp.id == -11.0):
                    print("Positron")
                if (pp.id == 11.0):
                    print("Electron")
                if (pp.id == -2.0):
                    print("Anti-Up")
                if (pp.id == 23.0):
                    print("Z Boson")
                if (pp.id == 22.0):
                    print("Photon")
                if (pp.id == 1.0):
                    print("Down")
                if (pp.id == 3.0):
                    print("Strange")
                if (pp.id == 21):
                    print("Gluon") 

                if (pp.id == 1000022 or pp.id == -1000022):
                    if pp.mother1 == 3:
                        n1E = pp.e
                        n1px = pp.px
                        n1py = pp.py
                        n1pz = pp.pz
                        n14 += [(n1E, n1px, n1py, n1pz)]
                    print("n1!")

                if (pp.id == 2000002):
                    print("up squark")
                if (pp.id == -2000002):
                    print("Anti up squark")

                #if( particle_count == 1 ):
                #    print("First particle information")
                #    print("px (GeV) =", pp.px)
                #    print("py (GeV) =", pp.py)
                #    print("pz (GeV) =", pp.pz)
                #    print("energy (GeV) =", pp.e)

                particle_count+=1
        """

    print(f"RUN NUMBER 0{fileVal}")
    print("Percent either 04:", anyCount04/eventCount*100)
    print("Percent both 04:", bothCount04/eventCount*100)
    print("Percent either 1:", anyCount1/eventCount*100)
    print("Percent both 1:", bothCount1/eventCount*100)
    print("Joining metric success:", successCount, (successCount/(eventCount*2))*100)
    print("CorrectChoice Count:", chosenCorrectlyCount, (chosenCorrectlyCount/(eventCount*2))*100)
    print("Same Mother Count:", sameMomCount, sameMomCount/eventCount*100)
    print("Picked other U count:", extraUCount, extraUCount/(eventCount*2)*100)
    print("BS METRIC:", bsMet, bsMet/(eventCount*2)*100)
    print("Both METRIC:", bothMet, bothMet/eventCount*100, "\n\n")

    #print(max(invM))

    #plt.hist(invM, bins= 10);
    #plt.hist(pTAm, bins= 10);

    #plt.show()

"""import pylhe
import math
import sys
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

import hist
from hist import Hist

file_name = '/home/joseph/MG_tutorial/MG5_aMC_v3_5_15/PPDP1/Events/run_02/unweighted_events.lhe'

eEn = []
ePx = []
ePy = []
ePz = []
piEn = []
piPx = []
piPy = []
piPz = []
invM = []
invME = []
invMPi = []
eMom1 = []
piMom1 = []
eMom2 = []
piMom2 = []


eventCount=1

for event in pylhe.read_lhe(file_name):

    particle_list = event.particles
    event_info = event.eventinfo

    #print("This is event", eventCount)
    #print(len(particle_list), "particles were made in this event")

    particle_count = 1
    for pp in particle_list:
        #print("Particle ", particle_count)
        #print("Particle ID number:", pp.id)
        #if (pp.id == 11.0):
            #print("Electron")

        if (pp.id == -11.0 and pp.status == 1):
            eMom1 += [pp.mother1]
            eMom2 += [pp.mother2]
            if (pp.mother1==4):
                print("POSITRON", (pp.e,pp.px,pp.py,pp.pz), "\n", eventCount)
                eEn += [pp.e]
                ePx += [pp.px]
                ePy += [pp.py]
                ePz += [pp.pz]
                invME += [np.sqrt((pp.e)**2-(pp.px)**2-(pp.py)**2-(pp.pz)**2)]
            #print("Positron")
        
        if (pp.id == 23.0):
            print("Z Boson")

        if (pp.id == 22.0):
            print("Photon")

        if (pp.id == 13.0):
            print("Muon")

        if (pp.id == -13.0):
            print("Anti-Muon")
        
        if (pp.id == -211 and pp.status == 1):
            piMom1 += [pp.mother1]
            piMom2 += [pp.mother2]
            if (pp.mother1==4):
                piEn += [pp.e]
                piPx += [pp.px]
                piPy += [pp.py]
                piPz += [pp.pz]
                invMPi += [np.sqrt((pp.e)**2-(pp.px)**2-(pp.py)**2-(pp.pz)**2)]
                print("PION-", (pp.e,pp.px,pp.py,pp.pz), "\n", eventCount)

        
        if( particle_count == 1 ):
            print("First particle information")
            print("px (GeV) =", pp.px)
            print("py (GeV) =", pp.py)
            print("pz (GeV) =", pp.pz)
            print("energy (GeV) =", pp.e)
        
        particle_count+=1

    eventCount+=1    

for i in range(50000):
    invM += [np.sqrt((eEn[i]+piEn[i])**2-(ePx[i]+piPx[i])**2-(ePy[i]+piPy[i])**2-(ePz[i]+piPz[i])**2)]


print("Pos AV", sum(invME)/len(invME))
print("Pi AV", sum(invMPi)/len(invMPi))
print("EM1", set(eMom1))
print("EM2", set(eMom2))
print("PiM1", set(piMom1))
print("PiM2", set(piMom2))

plt.hist(invM, bins= 100);
#plt.hist(pTAm, bins= 10);"""